# 3_1 Gold — Leads Activities Summary

Builds two Gold tables per spec Section 3:

1. **CLOSE_CRM_USERS_PROCESSED** — Deduplicated CRM user dimension with real names/emails
   - Replaces masked user hashes (`user_<32hex>`) with real human names
   - Used by LEADS_ACTIVITIES_SUMMARY for name enrichment and by SALES_DETAILS for setter/closer
   - Dedup: `USER_ID`, latest by `DATE_UPDATED DESC`

2. **LEADS_ACTIVITIES_SUMMARY** — Deduplicated activity records enriched with real CRM user names
   - Current flattened activity records with timestamp, type, resolved outcome, meeting fields, email/call attributes, and owner attribution
   - Dedup: `ACTIVITY_ID`, latest by `DATE_UPDATED DESC → INSERT_DATE DESC`

In [0]:
%sql
CREATE OR REPLACE TABLE crm_ingestion.gold.close_crm_users_processed AS (
    SELECT 
        csm_id,
        image,
        email,
        first_name,
        last_name,
        date_created,
        date_updated,
        bronze_insert_date,
        full_name AS csm_name,
        email_normalized AS csm_email
    FROM (
        SELECT
            id AS csm_id,
            image,
            email,
            first_name,
            last_name,
            date_created,
            date_updated,
            bronze_insert_date,
            ROW_NUMBER() OVER(PARTITION BY id ORDER BY date_updated DESC, bronze_insert_date DESC) AS rn,
            CONCAT_WS(' ',first_name, last_name) AS full_name,
            LOWER(TRIM(email)) AS email_normalized
        FROM
            crm_ingestion.silver.close_crm_users_processed      
)
    WHERE
        csm_id IS NOT NULL
    AND 
        rn = 1
)


In [0]:
%sql
SELECT 
    *
FROM
    crm_ingestion.gold.close_crm_users_processed
LIMIT 10

In [0]:
%sql
DESCRIBE TABLE crm_ingestion.silver.lead_activites_processed

In [0]:
%sql
CREATE OR REPLACE TABLE crm_ingestion.gold.leads_activities_summary AS (
    SELECT
        a.id AS activity_id,
        a.lead_id,
        a.type,
        a.activity_at,
        a.direction,
        a.status,
        a.disposition,
        a.subject,
        a.body_preview,
        a.text,
        a.starts_at,
        a.ends_at,
        a.outcome_id,
        a.custom_activity_type_id,
        a.created_by AS csm_id,
        COALESCE(u.csm_name, a.created_by_name) AS created_by_name,
        u.csm_email,
        a.date_created,
        a.date_updated,
        a.note,
        a.contact_id,
        a.organization_id,
        a.bronze_insert_date
    FROM (
        SELECT
            id,
            lead_id,
            type,
            activity_at,
            direction,
            status,
            disposition,
            subject,
            body_preview,
            text,
            starts_at,
            ends_at,
            outcome_id,
            custom_activity_type_id,
            created_by,
            created_by_name,
            date_created,
            date_updated,
            note,
            contact_id,
            organization_id,
            bronze_insert_date,
            ROW_NUMBER() OVER(PARTITION BY id ORDER BY date_updated DESC, bronze_insert_date DESC) AS rn
        FROM
            crm_ingestion.silver.lead_activites_processed
    ) a
    LEFT JOIN
        crm_ingestion.gold.close_crm_users_processed u
    ON
        a.created_by = u.csm_id
    WHERE
        a.id IS NOT NULL
    AND
        a.rn = 1
)
    

In [0]:
%sql
SELECT *
FROM crm_ingestion.gold.leads_activities_summary
LIMIT 100